# 02 · Fault tree and event tree analysis

The fire-protection fault tree shares two basic events (main power, common-cause failure of the electric pumps) between branches. This notebook shows why that matters.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
from offshore_risk import load_config
cfg = load_config()
from offshore_risk.fault_tree.protection import fire_protection_tree, firewater_subtree, fire_protection_basic_events
ft = fire_protection_tree(2, 1)
print(ft.to_text())
print('repeated events:', ft.repeated_events)

FIRE_PROTECTION_FAIL [OR]
├── FIREWATER_FAIL [OR]
│   ├── HEADER_FAIL
│   └── NO_FIREWATER_PUMP [AND]
│       ├── ALL_ELECTRIC_PUMPS_FAIL [AND]
│       │   ├── EPUMP_A_UNAVAILABLE [OR]
│       │   │   ├── EPUMP_A_FTS
│       │   │   ├── EPUMP_A_MAINT
│       │   │   ├── EPUMP_A_FTR
│       │   │   ├── POWER_LOSS  (repeated)
│       │   │   └── CCF_ELEC_PUMPS  (repeated)
│       │   └── EPUMP_B_UNAVAILABLE [OR]
│       │       ├── EPUMP_B_FTS
│       │       ├── EPUMP_B_MAINT
│       │       ├── EPUMP_B_FTR
│       │       ├── POWER_LOSS  (repeated)
│       │       └── CCF_ELEC_PUMPS  (repeated)
│       └── DPUMP_C_UNAVAILABLE [OR]
│           ├── DPUMP_C_FTS
│           ├── DPUMP_C_MAINT
│           └── DPUMP_C_FTR
├── DELUGE_VALVE_FAIL
└── NO_DELUGE_ACTIVATION [AND]
    ├── FIRE_DETECTION_FAIL
    └── MANUAL_ACTIVATION_FAIL
repeated events: ['CCF_ELEC_PUMPS', 'POWER_LOSS']


In [2]:
th = cfg.registry.at_quantile(1, 0.5)   # all parameters at their medians
rows = []
for pp in (0.05, 0.10, 0.60):
    be = fire_protection_basic_events(th, pp, 2, 1)
    fw = firewater_subtree(2, 1)
    rows.append({'P(power lost | fire)': pp, **{m: float(fw.probability(be, m)[0]) for m in ('independent', 'exact', 'rare_event', 'mcub', 'enumeration')}})
pd.DataFrame(rows)

,P(power lost | fire),independent,exact,rare_event,mcub,enumeration
0,0.05,0.002258,0.004038,0.004065,0.004060,0.004038
1,0.10,0.002659,0.006004,0.006057,0.006045,0.006004
2,0.60,0.016808,0.025670,0.025983,0.025800,0.025670


`independent` multiplies gate probabilities as if the two pump branches were independent; it underestimates firewater unavailability by roughly a factor of two here because the shared power supply is counted twice. `exact` (Shannon decomposition) agrees with brute-force `enumeration`.

In [3]:
be = fire_protection_basic_events(th, 0.10, 2, 1)
cs = [(' AND '.join(sorted(c)), float(np.prod([np.ravel(be[e])[0] for e in c]))) for c in ft.minimal_cut_sets()]
pd.DataFrame(cs, columns=['minimal cut set', 'probability']).sort_values('probability', ascending=False).head(10)

,minimal cut set,probability
0,DELUGE_VALVE_FAIL,0.013140
7,DPUMP_C_MAINT AND POWER_LOSS,0.002551
1,HEADER_FAIL,0.002000
6,DPUMP_C_FTS AND POWER_LOSS,0.001405
8,FIRE_DETECTION_FAIL AND MANUAL_ACTIVATION_FAIL,0.000257
5,DPUMP_C_FTR AND POWER_LOSS,0.000030
4,CCF_ELEC_PUMPS AND DPUMP_C_MAINT,0.000020
35,DPUMP_C_MAINT AND EPUMP_A_MAINT AND EPUMP_B_MAINT,0.000017
3,CCF_ELEC_PUMPS AND DPUMP_C_FTS,0.000011
26,DPUMP_C_FTS AND EPUMP_A_MAINT AND EPUMP_B_MAINT,0.000009


In [4]:
ft.importance(be).head(8)

,basic_event,probability,birnbaum,fussell_vesely,raw,rrw
0,DELUGE_VALVE_FAIL,0.013140,0.993740,0.675942,51.765638,3.085871
1,POWER_LOSS,0.100000,0.038805,0.200874,2.807866,1.251367
2,DPUMP_C_MAINT,0.025505,0.098626,0.130215,5.975226,1.149709
3,HEADER_FAIL,0.002000,0.982647,0.101735,51.765638,1.113257
4,DPUMP_C_FTS,0.014046,0.097480,0.070879,5.975226,1.076286
5,FIRE_DETECTION_FAIL,0.000858,0.294280,0.013073,16.220540,1.013247
6,MANUAL_ACTIVATION_FAIL,0.300000,0.000842,0.013073,1.030505,1.013247
7,DPUMP_C_FTR,0.000300,0.096139,0.001493,5.975226,1.001495


## Event tree for a hydrocarbon release
Branch probabilities for a medium gas release on PPA, at median parameters, derived from the reliability and fault-tree modules.

In [5]:
from offshore_risk.asset import AssetModel
from offshore_risk.simulation.derived import derive
from offshore_risk.event_tree import RELEASE_TREE
asset = AssetModel.from_config(cfg.asset); d = derive(th, asset); i = asset.index('PPA'); s = 1
p = {'p_detect': d.p_detect[0, s], 'p_isolate': d.p_isolate[0], 'p_ign': d.p_ign[0, s], 'p_ign_iso': d.p_ign[0, s] * th['ign_isolation_factor'][0],
     'p_exp': d.p_exp[0, s], 'p_fp_ok': 1 - d.pfd_fp[0, i]}
print({k: round(float(v), 5) for k, v in p.items()})
RELEASE_TREE.path_table(p)

{'p_detect': 0.84435, 'p_isolate': 0.99868, 'p_ign': 0.04536, 'p_ign_iso': 0.02268, 'p_exp': 0.1578, 'p_fp_ok': 0.98068}


,path,outcome,probability
0,Detected=Y -> Isolated=Y -> Ignition=N,controlled_release,8.241045e-01
1,Detected=Y -> Isolated=N -> Ignition=N,uncontrolled_release,1.067606e-03
2,Detected=N -> Ignition=N,uncontrolled_release,1.485929e-01
3,Detected=Y -> Isolated=Y -> Ignition=Y -> Expl...,minor_fire,1.579500e-02
4,Detected=Y -> Isolated=Y -> Ignition=Y -> Expl...,major_fire,3.111357e-04
5,Detected=Y -> Isolated=N -> Ignition=Y -> Expl...,major_fire,4.189629e-05
6,Detected=N -> Ignition=Y -> Explosion=N -> Fir...,major_fire,5.831261e-03
7,Detected=Y -> Isolated=Y -> Ignition=Y -> Expl...,explosion,2.959555e-03
8,Detected=Y -> Isolated=N -> Ignition=Y -> Expl...,explosion,7.850227e-06
9,Detected=N -> Ignition=Y -> Explosion=Y -> Fir...,explosion,1.092620e-03


In [6]:
pd.Series({k: float(v) for k, v in RELEASE_TREE.outcome_probabilities(p).items()})

controlled_release      0.824104
uncontrolled_release    0.149661
minor_fire              0.015795
major_fire              0.006184
explosion               0.004060
catastrophic            0.000196
dtype: float64

## Illustrative top event: major hydrocarbon fire on PPA within one year
AND of (at least one release), (ignition), (failure of prevention/mitigation) is exactly what the event tree already computes per release; the annual frequency is the release rate times the conditional probability of the major-fire outcomes.

In [7]:
rate = d.release_rate[0, i]
mix = {o: 0.0 for o in ('major_fire', 'explosion', 'catastrophic')}
for s in range(3):
    for gas, w in ((True, asset.gas_fraction[i]), (False, 1 - asset.gas_fraction[i])):
        pp = {'p_detect': d.p_detect[0, s], 'p_isolate': d.p_isolate[0], 'p_ign': d.p_ign[0, s], 'p_ign_iso': d.p_ign[0, s] * th['ign_isolation_factor'][0],
              'p_exp': d.p_exp[0, s] * (1 if gas else th['liquid_explosion_factor'][0]), 'p_fp_ok': 1 - d.pfd_fp[0, i]}
        op = RELEASE_TREE.outcome_probabilities(pp)
        for o in mix: mix[o] += d.size_probs[0, s] * w * float(op[o])
freq = rate * sum(mix.values())
print(f'release rate {rate:.3f}/yr; P(major fire or worse | release) {sum(mix.values()):.4f}; frequency {freq:.2e}/yr (1 in {1/freq:,.0f} yr)')

release rate 0.297/yr; P(major fire or worse | release) 0.0067; frequency 2.00e-03/yr (1 in 501 yr)
